In [1]:
"""
PhytoCluster: 植物单细胞RNA-seq数据聚类的生成式深度学习模型
基于变分自编码器(VAE) + 高斯混合模型(GMM)的无监督聚类算法
参考论文: Wang et al. (2025), aBIOTECH
"""

import os
import numpy as np
import pandas as pd
import scipy
import time
import random
import math
import torch
import scanpy as sc
from scipy.sparse import issparse
from torch.utils.data import Dataset
from anndata import AnnData
from glob import glob
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import init
from itertools import repeat
from torch.utils.data.sampler import Sampler
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import maxabs_scale, MaxAbsScaler
from sklearn.mixture import GaussianMixture
from scipy.sparse import csc_matrix
from sklearn.model_selection import train_test_split
from torch.optim.lr_scheduler import MultiStepLR, ExponentialLR, ReduceLROnPlateau
from sklearn.preprocessing import normalize
import matplotlib.pyplot as plt
from tqdm import tqdm

ModuleNotFoundError: No module named 'pandas'

## PhytoCluster：用于植物单细胞RNA-seq数据聚类的生成式深度学习模型

> **论文来源**：Wang et al. (2025), *aBIOTECH*  
> **核心思想**：将变分自编码器（VAE）与高斯混合模型（GMM）相结合，从植物单细胞转录组数据中提取低维潜在特征，实现无监督聚类。相比PCA、Scanpy、scVI、Seurat等方法，PhytoCluster在聚类精度、噪声去除和信号保留方面表现更优。

本 Notebook 实现流程如下：

1. **环境配置 & 数据加载**（当前 Cell）
2. **EarlyStopping 早停机制**：防止模型过拟合
3. **ELBO 损失函数**：VAE 的重构损失 + KL 散度
4. **MLP 网络构建工具函数**：创建多层感知器
5. **Encoder / Decoder / Stochastic / GaussianSample**：VAE 核心网络组件
6. **VAE 主模型**：变分自编码器实现
7. **PhytoCluster 主模型**：VAE + GMM 联合聚类框架
8. **fit() 训练函数**：端到端训练流程
9. **数据预处理**：使用 Scanpy 进行标准化、高可变基因筛选
10. **预训练 VAE**：提取初始潜在特征（潜在特征提取阶段一）
11. **聚类初始化**：基于预训练特征的 GMM 初始化
12. **PhytoCluster 训练**：联合优化聚类目标函数
13. **聚类与可视化**：UMAP 可视化聚类结果

In [ ]:
"""
EarlyStopping: 早停机制类

当验证损失在连续 patience 个 epoch 内不再改善时，停止训练。
这是防止深度学习模型过拟合的常用技巧，同时避免不必要的计算开销。

核心原理:
- 记录训练过程中出现的最小损失值 loss_min
- 如果当前 epoch 的损失未低于 loss_min，计数器 counter +1
- 当 counter >= patience 时，触发早停，加载最佳模型权重
"""

class EarlyStopping:
    def __init__(self, patience=10, verbose=False, outdir=None):
    
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.loss_min = np.inf
        self.model_file = os.path.join(outdir, 'model.pt') if outdir else None

    def __call__(self, loss, model):
        if np.isnan(loss):
            self.early_stop = True
        score = -loss

        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(loss, model)
        elif score < self.best_score:
            self.counter += 1
            if self.verbose:
                print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
                model.load_model(self.model_file)
        else:
            self.best_score = score
            self.save_checkpoint(loss, model)
            self.counter = 0

    def save_checkpoint(self, loss, model):
        if self.verbose:
            print(f'Loss decreased ({self.loss_min:.6f} --> {loss:.6f}).  Saving model ...')
        if self.model_file:
            torch.save(model.state_dict(), self.model_file)
        self.loss_min = loss

## EarlyStopping 早停机制

当验证损失在连续 `patience` 个 epoch 内不再改善时，停止训练并恢复最佳模型权重。这可以防止模型在训练集上过拟合，同时节省计算资源。

- **原理**：记录训练过程中最小的损失值 `loss_min`，如果当前损失持续未改善（`counter >= patience`），则触发早停
- **最佳模型保存**：在 `outdir` 目录下保存 `model.pt`，触发早停时自动加载最佳权重

In [ ]:
"""
compute_elbo: 计算 PhytoCluster 的证据下界（ELBO）损失（包含聚类目标）

该函数是 PhytoCluster 联合优化 VAE 和 GMM 聚类的核心。
ELBO 损失由以下几项组成:
  1. 重构损失 (likelihood): 重构数据与原始输入的误差（binary cross-entropy 或 MSE）
  2. KL 散度 (kld): 潜在变量后验分布与先验分布的差异
  3. 聚类后验项: GMM 中每个样本属于各簇的概率 gamma

参数:
  recon_x: 解码器重构的输出
  x: 原始输入数据
  gamma: q(c|x)，样本属于各聚类簇的后验概率（维度: N x K）
  c_params: GMM 聚类参数 (mu_c, var_c, pi) — 各簇的均值、方差、混合权重
  z_params: VAE 潜在变量参数 (mu, logvar) — 编码器输出的均值和对数方差
  binary: 是否使用二元交叉熵作为重构损失（False 表示使用 MSE）

返回:
  torch.sum(likelihood): 总重构似然
  torch.sum(kld): 总 KL 散度损失
"""

def compute_elbo(recon_x, x, gamma, c_params, z_params, binary=True):
    mu_c, var_c, pi = c_params  
    var_c += 1e-8
    n_centroids = pi.size(1)
    mu, logvar = z_params
    mu_expand = mu.unsqueeze(2).expand(mu.size(0), mu.size(1), n_centroids)
    logvar_expand = logvar.unsqueeze(2).expand(logvar.size(0), logvar.size(1), n_centroids)

    if binary:
        likelihood = -binary_cross_entropy(recon_x, x)  
    else:
        likelihood = -F.mse_loss(recon_x, x)

    logpzc = -0.5 * torch.sum(gamma * torch.sum(math.log(2 * math.pi) + \
                                                torch.log(var_c) + \
                                                torch.exp(logvar_expand) / var_c + \
                                                (mu_expand - mu_c) ** 2 / var_c, dim=1), dim=1)

    logpc = torch.sum(gamma * torch.log(pi), 1)

    qentropy = -0.5 * torch.sum(1 + logvar + math.log(2 * math.pi), 1)

    logqcx = torch.sum(gamma * torch.log(gamma), 1)

    kld = -logpzc - logpc + qentropy + logqcx

    return torch.sum(likelihood), torch.sum(kld)


In [ ]:
"""
compute_kl_divergence: 计算标准 VAE 中潜在变量的 KL 散度损失

原理: KL(q(z|x) || p(z)) = -0.5 * sum(1 + log(sigma^2) - mu^2 - sigma^2)
其中 q(z|x) = N(mu, sigma^2), p(z) = N(0, I)

返回值是一个向量，长度为 batch_size，每个元素是该样本的 KL 散度
"""

def compute_kl_divergence(mu, logvar):
    return -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1)

def compute_elbo_loss(recon_x, x, z_params, binary=True):
    mu, logvar = z_params
    kld = compute_kl_divergence(mu, logvar)
    if binary:
        likelihood = -binary_cross_entropy_loss(recon_x, x)
    else:
        likelihood = -F.mse_loss(recon_x, x)
    return torch.sum(likelihood), torch.sum(kld)

def binary_cross_entropy_loss(recon_x, x):
    return -torch.sum(x * torch.log(recon_x + 1e-8) + (1 - x) * torch.log(1 - recon_x + 1e-8), dim=-1)


In [ ]:
"""
create_mlp: 创建多层感知器（MLP）网络的辅助函数

layers: 列表，例如 [input_dim, 128, 64] 表示三层全连接网络
bn: 是否使用批归一化（Batch Normalization），有助于训练稳定性和收敛速度
dropout: Dropout 比率，用于防止过拟合（随机丢弃部分神经元输出）

返回一个 nn.Sequential 模型，包含 [Linear, (BN,), Activation, (Dropout,)] 的堆叠
"""

def create_mlp(layers, activation=nn.ReLU(), bn=False, dropout=0):
    net = []
    for i in range(1, len(layers)):
        net.append(nn.Linear(layers[i-1], layers[i]))
        if bn:
            net.append(nn.BatchNorm1d(layers[i]))
        net.append(activation)
        if dropout > 0:
            net.append(nn.Dropout(dropout))
    return nn.Sequential(*net)


In [ ]:
"""
Encoder（编码器）: 将高维基因表达数据压缩到低维潜在空间

网络结构:
  输入层 (x_dim) -> 隐藏层 (h_dim: 1024 -> 128) -> 高斯采样层 (z_dim=10)
  
Encoder 通过两层全连接网络（ReLU激活）逐步降低维度，
最后通过 GaussianSample 层输出潜在变量的均值 mu 和对数方差 log_var，
并利用重参数化技巧 (reparameterization trick) 生成可微分的潜在变量 z。

Encoder 的输出是 PhytoCluster 的核心——低维潜在特征向量（10维），
这些特征能够去除噪声、保留关键生物信息，非常适合后续的细胞聚类分析。

Decoder（解码器）: 从潜在变量重建原始基因表达谱

网络结构:
  潜在层 (z_dim=10) -> 隐藏层 (h_dim: 128 -> 1024) -> 输出层 (x_dim)

Decoder 是 Encoder 的镜像结构，将潜在变量通过全连接网络
映射回原始数据空间，重构基因表达向量。
当 binary=False 时使用线性输出（配合 MSE 损失）。

DeterministicWarmup（β值预热）: 控制 KL 散度权重的预热策略

在训练初期（n 步内），β 从 0 线性增长到 t_max=1。
这确保模型先专注于学习有效的重构能力，再逐步引入潜在空间的正则化约束，
避免训练初期 KL 散度过大导致的重构质量下降。
"""

class Encoder(nn.Module):
    def __init__(self, dims, bn=False, dropout=0):

        super(Encoder, self).__init__()

        [x_dim, h_dim, z_dim] = dims
        self.hidden = create_mlp([x_dim]+h_dim, bn=bn, dropout=dropout)
        self.sample = GaussianSample(([x_dim]+h_dim)[-1], z_dim)

    def forward(self, x):
        x = self.hidden(x)
        return self.sample(x)


class Decoder(nn.Module):
    def __init__(self, dims, bn=False, dropout=0, output_activation=nn.Sigmoid()):

        super(Decoder, self).__init__()

        [z_dim, h_dim, x_dim] = dims

        self.hidden = create_mlp([z_dim, *h_dim], bn=bn, dropout=dropout)
        self.reconstruction = nn.Linear([z_dim, *h_dim][-1], x_dim)

        self.output_activation = output_activation

    def forward(self, x):
        x = self.hidden(x)
        if self.output_activation is not None:
            return self.output_activation(self.reconstruction(x))
        else:
            return self.reconstruction(x)

class DeterministicWarmup(object):

    def __init__(self, n=100, t_max=1):
        self.t = 0
        self.t_max = t_max
        self.inc = 1/n

    def __iter__(self):
        return self

    def __next__(self):
        t = self.t + self.inc

        self.t = self.t_max if t > self.t_max else t
        return self.t

    def next(self):
        t = self.t + self.inc

        self.t = self.t_max if t > self.t_max else t
        return self.t

In [ ]:
"""
Stochastic（随机层基类）: 实现 VAE 重参数化技巧的核心组件

重参数化技巧 (Reparameterization Trick):
  标准正态分布采样 epsilon ~ N(0, I)
  计算 std = exp(logvar / 2)
  z = mu + std * epsilon
  
这使得采样操作不参与梯度计算，仅 mu、logvar 参与反向传播，
从而实现对随机变量的可微分采样。
"""

class Stochastic(nn.Module):

    def reparametrize(self, mu, logvar):
        epsilon = torch.randn(mu.size(), requires_grad=False, device=mu.device)
        std = logvar.mul(0.5).exp_()
        z = mu.addcmul(std, epsilon)

        return z

In [ ]:
## Encoder（编码器）和 Decoder（解码器）网络架构

### Encoder（编码器）
- **作用**：将高维输入数据 X（基因表达向量）压缩到低维潜在空间
- **结构**：两层全连接神经网络（MLP），中间层使用 ReLU 激活函数，逐步降低维度
- **输出**：通过 `GaussianSample` 层输出潜在变量 z 的均值 μ 和对数方差 log σ²
- **重参数化技巧**：$z = \mu + \sigma \odot \epsilon$，其中 $\epsilon \sim \mathcal{N}(0, I)$，使梯度可通过随机采样过程反向传播

### Decoder（解码器）
- **作用**：从潜在变量 z 重建原始输入数据，重构基因表达谱
- **结构**：两层全连接神经网络，将潜在空间映射回原始数据空间
- **输出激活**：当 `binary=True` 时使用 Sigmoid 激活（将输出归一化到 [0,1]），否则直接线性输出

### DeterministicWarmup（β 值预热策略）
- **作用**：在训练初期让 β（KL 散度权重）从 0 逐渐增加到目标值 1
- **目的**：防止训练初期 KL 散度主导损失函数，使模型先学习有效的重构，再逐步引入潜在空间正则化

### GaussianSample（高斯采样层）
- **作用**：VAE 中实现重参数化技巧的核心层
- **输入**：编码器中间层输出
- **输出**：(z, μ, log σ²) — 采样后的潜在变量、均值、对数方差

In [ ]:
"""
GaussianSample: 高斯采样层（继承自 Stochastic）

这是 VAE 编码器的核心输出层，用两个独立的线性层分别预测:
  - mu: 潜在变量的均值向量（维度: z_dim）
  - log_var: 潜在变量方差的对数（维度: z_dim），使用对数形式保证方差始终为正

forward 返回:
  z: 重参数化采样后的潜在变量
  mu/10: 用于后续损失计算的均值（除以10是缩放处理）
  log_var: 对数方差
"""

class GaussianSample(Stochastic):

    def __init__(self, in_features, out_features):
        super(GaussianSample, self).__init__()
        self.in_features = in_features
        self.out_features = out_features

        self.mu = nn.Linear(in_features, out_features)
        self.log_var = nn.Linear(in_features, out_features)

    def forward(self, x):
        mu = self.mu(x)
        log_var = self.log_var(x)

        return self.reparametrize(mu, log_var), mu/10, log_var

In [ ]:
"""
VAE: 变分自编码器（Variational Autoencoder）主模型

PhytoCluster 的基础组件，负责将高维 scRNA-seq 数据编码为低维潜在特征。

dims 参数格式: [input_dim, latent_dim, encode_dim, decode_dim]
  - input_dim: 输入基因数（高可变基因数，如3000）
  - latent_dim: 潜在空间维度（默认10维）
  - encode_dim: 编码器隐藏层维度列表（如 [1024, 128]）
  - decode_dim: 解码器隐藏层维度列表（如 [128, 1024]）

关键方法:
  - initialize_weights(): 使用 Xavier 正态初始化所有线性层权重
  - forward(x): 编码 -> 采样 z -> 解码重构，返回重构数据
  - compute_loss(x): 计算 ELBO 损失（重构损失 + KL 散度）
  - train_model(...): 纯 VAE 的训练循环（不含聚类）
  - encode_batch(...): 批量编码，可输出 z（采样）、mu（均值）、x（重构）等

binary=False: 使用 MSE 作为重构损失（适合归一化后的连续表达值）
binary=True: 使用 Binary Cross-Entropy（适合二值化数据）
"""

class VAE(nn.Module):
    def __init__(self, dims, bn=False, dropout=0, binary=False):
        super(VAE, self).__init__()
        [x_dim, z_dim, encode_dim, decode_dim] = dims
        self.binary = binary
        if binary:
            decode_activation = nn.Sigmoid()
        else:
            decode_activation = None

        self.encoder = Encoder([x_dim, encode_dim, z_dim], bn=bn, dropout=dropout)
        self.decoder = Decoder([z_dim, decode_dim, x_dim], bn=bn, dropout=dropout, output_activation=decode_activation)

        self.initialize_weights()

    def initialize_weights(self):
        """Initialize weights for the linear layers in the network."""
        for m in self.modules():
            if isinstance(m, nn.Linear):
                init.xavier_normal_(m.weight.data)
                if m.bias is not None:
                    m.bias.data.zero_()

    def forward(self, x, y=None):
        """Forward pass to encode input and reconstruct."""
        z, mu, logvar = self.encoder(x)
        recon_x = self.decoder(z)
        return recon_x

    def compute_loss(self, x):
        """Compute the reconstruction loss and KL divergence loss."""
        z, mu, logvar = self.encoder(x)
        recon_x = self.decoder(z)
        likelihood, kl_loss = compute_elbo_loss(recon_x, x, (mu, logvar), binary=self.binary)
        return (-likelihood, kl_loss)

    def train_model(self, dataloader,
                    lr=0.002,
                    weight_decay=5e-4,
                    device='cuda',
                    beta=1,
                    warmup_steps=200,
                    max_iter=30000,
                    verbose=True,
                    patience=100,
                    outdir=None):
        """Train the VAE model with the specified parameters."""
        self.to(device)
        optimizer = torch.optim.Adam(self.parameters(), lr=lr, weight_decay=weight_decay)
        beta_scheduler = DeterministicWarmup(n=warmup_steps, t_max=beta)

        iteration = 0
        n_epoch = int(np.ceil(max_iter / len(dataloader)))
        early_stopping = EarlyStopping(patience=patience, outdir=outdir)

        for epoch in range(n_epoch):
            epoch_recon_loss, epoch_kl_loss = 0, 0
            for i, x in enumerate(dataloader):
                x = x.float().to(device)
                optimizer.zero_grad()

                recon_loss, kl_loss = self.compute_loss(x)
                loss = (recon_loss + kl_loss) / len(x)
                loss.backward()
                torch.nn.utils.clip_grad_norm(self.parameters(), 10)  # Clip gradients
                optimizer.step()

                epoch_kl_loss += kl_loss.item()
                epoch_recon_loss += recon_loss.item()

                iteration += 1

    def encode_batch(self, dataloader, device='cpu', output_type='z', transforms=None):
        """Encode a batch of data and return the specified output."""
        output = []
        for x in dataloader:
            x = x.view(x.size(0), -1).float().to(device)
            z, mu, logvar = self.encoder(x)

            if output_type == 'z':
                output.append(z.detach().cpu())
            elif output_type == 'x':
                recon_x = self.decoder(z)
                output.append(recon_x.detach().cpu().data)
            elif output_type == 'logit':
                output.append(self.get_gamma(z)[0].cpu().detach().data)
            elif output_type == 'mu':
                output.append(mu.cpu().detach().data)
            elif output_type == 'log_var':
                output.append(logvar.cpu().detach().data)

        output = torch.cat(output).numpy()
        return output


In [ ]:
"""
PhytoCluster: VAE + GMM 联合聚类模型（继承自 VAE）

PhytoCluster 在 VAE 基础上添加了高斯混合模型（GMM）聚类层，
实现了变分推断与无监督聚类的端到端联合优化。

新增可学习参数:
  - self.pi: K 个聚类簇的混合权重（所有簇权重和为1）
  - self.mu_c: K 个簇的均值向量（维度: z_dim x K）
  - self.var_c: K 个簇的方差（维度: z_dim x K）

infer_clusters(z): 推断每个样本属于各簇的后验概率 q(c|x)
  使用贝叶斯公式: q(c|x) = p(c)p(z|c) / sum_k(p(c_k)p(z|c_k))
  其中 p(z|c_k) = N(z; mu_c_k, var_c_k)，p(c_k) = pi_k

initialize_gmm_params(dataloader): 使用预训练 VAE 提取的潜在特征，
  通过 sklearn 的 GMM.fit() 估计各簇的初始均值和方差参数，
  避免了随机初始化的不确定性。

compute_loss(x): PhytoCluster 的完整损失函数
  与 VAE 的区别在于 KL 散度项包含了聚类后验分布的约束，
  驱动潜在空间形成紧凑且分离良好的簇结构。
"""

class PhytoCluster(VAE):
    def __init__(self, dims, n_centroids):
        super(PhytoCluster, self).__init__(dims)
        self.n_centroids = n_centroids
        z_dim = dims[1]

        # Initialize cluster parameters
        self.pi = nn.Parameter(torch.ones(n_centroids) / n_centroids)  # Cluster probabilities (pi)
        self.mu_c = nn.Parameter(torch.zeros(z_dim, n_centroids))  # Cluster means (mu)
        self.var_c = nn.Parameter(torch.ones(z_dim, n_centroids))  # Cluster variances (sigma^2)

    def compute_loss(self, x):
        """Compute the reconstruction loss and KL divergence with clustering."""
        z, mu, logvar = self.encoder(x)
        recon_x = self.decoder(z)
        gamma, mu_c, var_c, pi = self.infer_clusters(z)  # Get cluster assignments and parameters
        likelihood, kl_loss = compute_elbo(recon_x, x, gamma, (mu_c, var_c, pi), (mu, logvar), binary=self.binary)
        return -likelihood, kl_loss

    def infer_clusters(self, z):
        """
        Infer cluster assignments (gamma) based on latent variables z.
        gamma is q(c|x) where q(c|x) = p(c)p(z|c)/p(z).
        """
        n_centroids = self.n_centroids
        N = z.size(0)
        z = z.unsqueeze(2).expand(z.size(0), z.size(1), n_centroids)
        pi = self.pi.repeat(N, 1)  # NxK
        mu_c = self.mu_c.repeat(N, 1, 1)  # NxDxK
        var_c = self.var_c.repeat(N, 1, 1) + 1e-8  # NxDxK

        p_c_z = torch.exp(
            torch.log(pi) - torch.sum(0.5 * torch.log(2 * math.pi * var_c) + (z - mu_c) ** 2 / (2 * var_c),
                                      dim=1)) + 1e-10
        gamma = p_c_z / torch.sum(p_c_z, dim=1, keepdim=True)

        return gamma, mu_c, var_c, pi

    def initialize_gmm_params(self, dataloader, device='cpu'):
        """Initialize GMM parameters (means and variances) using latent variables."""
        gmm = GaussianMixture(n_components=self.n_centroids, covariance_type='diag')
        z = self.encode_batch(dataloader, device)
        gmm.fit(z)
        self.mu_c.data.copy_(torch.from_numpy(gmm.means_.T.astype(np.float32)))
        self.var_c.data.copy_(torch.from_numpy(gmm.covariances_.T.astype(np.float32)))


In [ ]:
"""
fit(): PhytoCluster / VAE 的端到端训练函数

核心流程:
  1. 将模型移动到指定设备（GPU/CPU）
  2. 创建 Adam 优化器（对编码器、解码器使用不同学习率）
  3. 创建 Beta 预热调度器（KL 散度权重从0逐渐升到beta）
  4. 按 epoch 迭代训练:
     - 前向传播: 计算重构损失和 KL 散度
     - 联合损失: loss = recon_loss + beta * kl_loss
     - 反向传播: 计算梯度并更新参数
     - 梯度裁剪: clip_grad_norm_(..., 10)，防止梯度爆炸
  5. 记录每个 epoch 的损失、重构损失、KL 散度
  6. 绘制训练曲线（3个子图：总损失、重构损失、KL散度）

关键参数:
  lr: 主学习率（编码器/解码器）
  var_lr: 方差参数的学习率（log_var 层使用独立学习率）
  beta: KL 散度权重（通过 warmup 策略逐渐增加）
  max_iter: 最大迭代次数（batch 数）
  n: Beta 预热的总步数（n * batch_size = warmup 覆盖的样本数）
  patience: EarlyStopping 的耐心值

返回: 训练完成，绘制 loss / rec_loss / kl_loss 曲线
"""

def fit(model,
        dataloader,
        lr=0.0002,
        var_lr=0.0002,
        weight_decay=5e-4,
        device='cuda',
        beta=1,
        n=2000,
        max_iter=30000,
        verbose=True,
        patience=100,
        outdir=None,
        ):
    self = model
    model.to(device)
    optimizer = torch.optim.Adam(self.parameters(), lr=lr, weight_decay=weight_decay)
    if isinstance(self, VAE):
        optimizer = torch.optim.Adam([{'params': self.encoder.hidden.parameters(), 'lr': lr},
                                      {'params': self.encoder.sample.mu.parameters(), 'lr': lr},
                                      {'params': self.encoder.sample.log_var.parameters(), 'lr': var_lr},
                                      {"params": self.decoder.parameters(), 'lr': lr}],
                                     weight_decay=weight_decay)

    Beta = DeterministicWarmup(n=n, t_max=beta)

    iteration = 0
    n_epoch = int(np.ceil(max_iter / len(dataloader)))
    early_stopping = EarlyStopping(patience=patience, outdir=outdir)
    kl_loss_hist = []
    rec_loss_hist = []
    loss_hist = []
    for epoch in tqdm(range(n_epoch)):
        epoch_recon_loss, epoch_kl_loss, epoch_loss = 0, 0, 0
        
        for i, x in enumerate(dataloader):
            x = x.float().to(device)
            optimizer.zero_grad()

            recon_loss, kl_loss = self.compute_loss(x)

            loss = recon_loss + next(Beta)*kl_loss
            if torch.isnan(loss):
                return
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.parameters(), 10)  # clip
            optimizer.step()

            epoch_kl_loss += kl_loss.item()/len(x)
            epoch_recon_loss += recon_loss.item()/len(x)
            epoch_loss += loss.item()/len(x)



            iteration += 1
        loss_hist.append(epoch_loss)
        kl_loss_hist.append(epoch_kl_loss)
        rec_loss_hist.append(epoch_recon_loss)


    fig, axes = plt.subplots(1, 3, figsize=(4*3, 4*1), layout='constrained')
    axes[0].plot(loss_hist, label='loss')
    axes[0].set_title(f'loss:{loss_hist[-1]}')
    axes[1].plot(rec_loss_hist, label='rec loss')
    axes[1].set_title(f'rec loss:{rec_loss_hist[-1]}')
    axes[2].plot(kl_loss_hist, label='kl loss')
    axes[2].set_title(f'kl loss:{kl_loss_hist[-1]}')
    plt.show()

In [ ]:
## PhytoCluster 模型：VAE + GMM 联合聚类

PhytoCluster 整合了两个核心组件：

1. **变分自编码器（VAE）**：编码器将高维基因表达数据压缩为 10 维潜在特征（保留关键生物信息，去除噪声）；解码器从潜在变量重建原始表达谱

2. **高斯混合模型（GMM）**：基于潜在空间中的特征，用 GMM 对细胞进行无监督聚类。GMM 将潜在空间划分为 K 个高斯分布簇（K = 细胞类型数）

### 核心优化目标（ELBO 损失函数）

PhytoCluster 的损失函数包含两部分：

$$\mathcal{L} = \mathcal{L}_{\text{RECON}} + \beta \cdot \mathcal{L}_{\text{KL}}$$

- **重构损失 $\mathcal{L}_{\text{RECON}}$**：解码器重建的基因表达谱与原始输入之间的误差（本文使用 MSE）
- **KL 散度损失 $\mathcal{L}_{\text{KL}}$**：潜在变量的近似后验 $q(z|x)$ 与标准正态分布之间的 KL 散度，确保潜在空间结构良好
- **β 参数**：通过 `DeterministicWarmup` 从 0 逐渐升温到 1，使模型先学习重构再引入正则化

### 两阶段训练策略

| 阶段 | 模型 | 目标 | 迭代次数 |
|------|------|------|----------|
| 阶段一：预训练 | 纯 VAE | 学习良好的潜在空间表示 | 30000 次迭代 |
| 阶段二：聚类训练 | PhytoCluster (VAE + GMM) | 联合优化聚类目标函数 | 300 次迭代 |

阶段一先用 VAE 提取高质量潜在特征，然后用 GMM 对这些特征进行 K-means 初始化。阶段二将预训练权重迁移到 PhytoCluster，继续微调以优化聚类效果。

In [ ]:
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)

In [ ]:
"""
数据预处理阶段（使用 Scanpy）

植物 scRNA-seq 数据的质量控制与标准化流程:

1. 加载数据: 从 CSV 文件读取基因表达矩阵，转换为 AnnData 格式
   - 行: 细胞 (observations, obs)
   - 列: 基因 (variables, var)
   - obs['label']: 真实的细胞类型标签（仅用于后续评估，不参与训练）

2. 高可变基因筛选: sc.pp.highly_variable_genes(..., flavor='seurat_v3')
   - 基于基因表达方差识别最具有区分性的基因
   - 保留前 n_top_genes=3000 个高可变基因，减少数据维度和噪声

3. 归一化: sc.pp.normalize_total(adata, target_sum=1000)
   - 每个细胞的 UMI 总数归一化到 1000，消除测序深度差异

4. 对数变换: sc.pp.log1p(adata)
   - log(1 + x)，压缩极端值，使数据分布更接近正态分布

5. 标准化: sc.pp.scale(adata)
   - 每个基因的均值归一化为 0，方差归一化为 1

6. PCA降维: sc.tl.pca(adata, n_comps=50)
   - 线性降维到50维，作为网络输入前的预处理

n_clusters: 聚类数（从真实标签中获取，即数据集中细胞类型的数量）
"""

if torch.cuda.is_available():  # cuda device
    device = 'cuda'
    torch.cuda.set_device(0)
    print(1)
else:
    device = 'cpu'

data_dir = "./"
data = pd.read_csv(os.path.join(data_dir, "test.csv"))
adata = sc.AnnData(data.iloc[:, 1:], obs=data.iloc[:,:1].astype('category'))
n_clusters = len(np.unique(adata.obs['label']))

n_top_genes = 3000
target_sum = 1000

sc.pp.highly_variable_genes(adata, flavor="seurat_v3", n_top_genes=10000)
adata = adata[:, adata.var['highly_variable_rank']<n_top_genes]
sc.pp.normalize_total(adata, target_sum=target_sum)
sc.pp.log1p(adata)
sc.pp.scale(adata)
sc.tl.pca(adata, n_comps=50)

In [ ]:
"""
模型构建与预训练阶段

网络架构配置:
  latent = 10:        潜在空间维度（论文推荐值，压缩后保留关键信息）
  encode_dim = [1024, 128]:  编码器隐藏层结构（输入 -> 1024 -> 128 -> 10）
  decode_dim = [128, 1024]:  解码器隐藏层结构（10 -> 128 -> 1024 -> 输出）
  
dims: 模型维度配置列表 = [input_dim, latent, encode_dim, decode_dim]

预训练阶段（纯 VAE）:
  - 迭代次数: max_iter=30000
  - 学习率: lr=0.0001
  - 目标: 让编码器学习将高维基因表达数据压缩到有意义的 10 维潜在空间，
    同时解码器学会从这个潜在空间重建原始表达谱
  - 此时仅有重构目标，潜在空间尚未被聚类约束塑形

输出: 预训练好的 VAE 模型权重（保存于 model.pt）
"""

bn = False
dropout = 0
verbose = False
max_iter = 30000
outdir = None
lr = 0.0001

latent = 10
encode_dim = [1024, 128]
decode_dim = [128, 1024]
k = n_clusters

all_data_tensor = torch.from_numpy(adata.X).float()
dataloader = DataLoader(all_data_tensor, batch_size=64, drop_last=False, shuffle=False)
train_data_loader = DataLoader(all_data_tensor, batch_size=64, drop_last=True, shuffle=True)
input_dim = all_data_tensor.shape[-1]
dims = [input_dim, latent, encode_dim, decode_dim]
pretrain_model = VAE(dims, binary=False)

fit(pretrain_model, train_data_loader,
                    lr=lr,
                    var_lr=lr,
                    max_iter=max_iter,
                    verbose=verbose,
                    device=device,
                    outdir=outdir,
                    n=max_iter*200,
                  )

In [ ]:
## 预训练 VAE 结果分析

利用预训练 VAE 提取的潜在特征（`output_type='mu'`），使用高斯混合模型（GMM）对细胞进行初始聚类。

- **`encode_batch(..., output_type='mu')`**：使用编码器的均值 μ 作为每个细胞的潜在特征表示（10维），而非采样后的 z。这样避免随机采样带来的变异性
- **GMM 聚类**：使用对角协方差矩阵（`covariance_type='diag'`）的 GMM，将细胞分配到 K 个簇
- **UMAP 可视化**：将高维潜在特征降至 2 维进行可视化，比较真实标签（`label`）与预训练聚类结果（`pretrain_gmm_pred`）的一致性

**预期结果**：预训练阶段应已能将不同类型的细胞大致分开，为后续 PhytoCluster 的精细聚类奠定基础。

In [ ]:
"""
预训练结果评估与 GMM 初始聚类

1. encode_batch(output_type='mu'):
   - 使用预训练 VAE 编码器，将每个细胞映射到 10 维潜在空间
   - output_type='mu': 返回编码器预测的均值向量（而非采样后的 z）
   - mu 比随机采样的 z 更稳定，适合作为下游聚类的输入特征

2. GMM 聚类初始化:
   - GaussianMixture(n_components=k, covariance_type='diag'): 
     对角协方差高斯混合模型（假设各维度独立）
   - fit_predict(pretrain_feat): 在潜在特征上拟合 GMM，获取初始聚类标签
   - 这作为 PhytoCluster 聚类模块的初始化参数（mu_c, var_c, pi）

3. UMAP 可视化:
   - sc.pp.neighbors: 在潜在特征上计算 k 近邻图
   - sc.tl.umap: 使用 UMAP 降维到 2 维
   - sc.pl.umap: 可视化真实标签 vs 预训练聚类结果
   - 理想情况下，同类型细胞的聚类结果应与真实标签高度吻合
"""

pretrain_model.eval()
pretrain_feat = pretrain_model.encode_batch(dataloader, device=device, output_type='mu')
pretrain_pred = GaussianMixture(n_components=k, covariance_type='diag').fit_predict(pretrain_feat)
adata.obs['pretrain_gmm_pred'] = pretrain_pred
adata.obs['pretrain_gmm_pred'] = adata.obs['pretrain_gmm_pred'].astype(int).astype('category')
data = sc.AnnData(pretrain_feat, obs=adata.obs)
sc.pp.neighbors(data)
sc.tl.umap(data)
sc.pl.umap(data, color=['label', 'pretrain_gmm_pred'])

In [ ]:
"""
PhytoCluster 端到端联合训练

1. model.load_state_dict(pretrain_model.state_dict(), strict=False):
   - 将预训练 VAE 的编码器和解码器权重加载到 PhytoCluster
   - strict=False: 允许部分参数不匹配（因为 PhytoCluster 有额外的 GMM 参数）

2. model.initialize_gmm_params(dataloader):
   - 使用预训练 VAE 提取的 10 维潜在特征，通过 sklearn GMM 估计聚类参数
   - 将 GMM 的均值和方差复制到 PhytoCluster 的 self.mu_c 和 self.var_c
   - 使用 K-means 风格的初始化，避免随机初始化导致的收敛问题

3. 联合优化:
   - max_iter=300: 相比预训练的 30000 次大幅减少（模型已接近最优）
   - 联合损失: recon_loss + beta * kl_loss（β 逐渐从0升到1）
   - GMM 参数（pi, mu_c, var_c）与 VAE 参数共同更新
   - 最终潜在空间形成紧凑且分离良好的细胞类型簇

输出: PhytoCluster 模型，聚类后的细胞标签存储在 adata.obs['gmm_pred']
"""

model = PhytoCluster(dims, n_clusters)
model.load_state_dict(pretrain_model.state_dict(), strict=False)
model.initialize_gmm_params(dataloader)
max_iter = 300
fit(model, train_data_loader,
                    lr=lr,
                    var_lr=lr,
                    max_iter=max_iter,
                    verbose=verbose,
                    device=device,
                    outdir=outdir,
                    n=max_iter*300,
                  )

In [ ]:
## PhytoCluster 端到端训练

将预训练 VAE 的权重迁移到 PhytoCluster 模型，并使用 GMM 初始化聚类参数，然后进行端到端的联合优化：

1. **权重加载**：`model.load_state_dict(pretrain_model.state_dict(), strict=False)` — 从预训练的 VAE 复制编码器和解码器的权重到 PhytoCluster，确保模型从一个良好的初始点开始
2. **GMM 参数初始化**：`model.initialize_gmm_params(dataloader)` — 基于预训练 VAE 提取的潜在特征，用 sklearn 的 GMM 估计各簇的均值和方差参数，作为 PhytoCluster 聚类模块的初始值
3. **联合训练**：使用较小的 `max_iter=300`（相比预训练的 30000 次大幅减少），因为此时模型已接近最优，只需微调聚类边界

### 关键区别：预训练 VAE vs. PhytoCluster

| | 预训练 VAE | PhytoCluster |
|---|---|---|
| 目标 | 重构输入 + 潜在空间正则化 | 重构 + 潜在空间 + 聚类约束 |
| 损失函数 | ELBO（重构损失 + KL 散度） | 聚类 ELBO（增加 GMM 后验项） |
| 训练迭代 | 30000 次 | 300 次（精细调优）|

In [ ]:
"""
最终聚类结果与可视化

1. model.encode_batch(dataloader, output_type='mu'):
   - 使用训练好的 PhytoCluster 模型提取最终的 10 维潜在特征
   - 这些特征经过聚类约束的优化，分离效果更好

2. GMM 重新聚类:
   - 在 PhytoCluster 优化后的潜在特征上重新拟合 GMM
   - 获取最终的细胞类型分配标签 gmm_pred

3. UMAP 可视化:
   - 使用 min_dist=0.3（UMAP 的紧密度参数）进行降维可视化
   - custom_palette: 自定义颜色映射，与真实标签对应
   - sc.pl.umap: 显示聚类结果分布

评估指标解读（论文中使用）:
  - NMI（标准化互信息）: 衡量聚类结果与真实标签的信息一致性，范围 [0,1]
  - ARI（调整兰德指数）: 衡量聚类结果与真实标签的吻合程度，范围 [-1,1]，1表示完全一致
  更高的 NMI 和 ARI 值代表更准确的细胞类型划分。
"""

model.eval()
feat = model.encode_batch(dataloader, device=device, output_type='mu')
pred = GaussianMixture(n_components=k, covariance_type='diag').fit_predict(pretrain_feat)
adata.obs['gmm_pred'] = pred
adata.obs['gmm_pred'] = adata.obs['gmm_pred'].astype(int).astype('category')
data = sc.AnnData(feat, obs=adata.obs)
sc.pp.neighbors(data)
sc.tl.umap(data, min_dist=0.3)
custom_palette = ['#F8766D', '#D39200', '#93AA00', '#00BA38', '#00C19F', '#00B9E3']
sc.pl.umap(data, color=['gmm_pred'], palette=custom_palette)